# Chapter 18 — Representation Is Part of Context

## Question

**If two contexts contain the same underlying facts in different forms, are they the same context?**

Structural answer: no. Every condition below is rendered deterministically from one canonical manifest — no model call in condition construction — and every claim is a mechanical property, never a comprehension score. If one form dominated every operation, the fixture would be rebuilt as biased.

## Setup — one canonical manifest with traps

Near-identical identifiers (01947 vs 01974), zero vs null vs absent, a directional relation, dual-source agreement, and shared outcomes with divergent rationales.

In [ ]:
MANIFEST = {
    'migration_id': '01947',
    'deploy_id': '01974',
    'backend': 'PostgreSQL',
    'backend_sources': ['ADR-007', 'staging-log'],
    'status': 'blocked',
    'blocking_test': 'suite-A',
    'rejected_alternative': 'SQLite',
    'rejection_reason': 'loses serialisable transactions under batch writes',
    'second_rejected': 'Redis',
    'second_reason': 'no transaction support at all',
    'owner': 'migrations-team',
    'retry_count': 0,
    'deadline': None,
    'relation': ('migration-01947', 'depends_on', 'suite-A'),
}
print(f'{len(MANIFEST)} canonical fields; optional_field deliberately absent.')

## Renderers — deterministic, no model anywhere

In [ ]:
import json as _json

def r_prose(m):
    return ('The migration uses PostgreSQL. Its identifier is 01947. A separate deploy 01974 is unrelated. '
            'It remains blocked by test suite-A. The team rejected the SQLite fallback because it loses '
            'serialisable transactions under batch writes. Redis was also rejected: no transaction support at all. '
            'Both ADR-007 and the staging log confirm PostgreSQL. Retry count is 0 and no deadline is set. '
            'Migration 01947 depends on suite-A.')

def r_keyvalue(m):
    return '\n'.join(['migration_id: 01947', 'deploy_id: 01974', 'backend: PostgreSQL',
                       'backend_sources: ADR-007, staging-log', 'status: blocked',
                       'blocking_test: suite-A', 'rejected_alternative: SQLite',
                       'rejection_reason: loses serialisable transactions under batch writes',
                       'second_rejected: Redis', 'second_reason: no transaction support at all',
                       'owner: migrations-team', 'retry_count: 0', 'deadline: null',
                       'relation: depends_on suite-A'])

def record_dict(m):
    return {'migration_id': m['migration_id'], 'deploy_id': m['deploy_id'], 'backend': m['backend'],
            'backend_sources': m['backend_sources'], 'status': m['status'],
            'blocking_test': m['blocking_test'], 'rejected_alternative': m['rejected_alternative'],
            'rejection_reason': m['rejection_reason'], 'owner': m['owner'],
            'retry_count': m['retry_count'], 'deadline': m['deadline'],
            'relation': {'subject': m['relation'][0], 'predicate': m['relation'][1], 'object': m['relation'][2]}}

def r_json(m):
    return _json.dumps(record_dict(m), indent=1)

def r_xml(m):
    r = record_dict(m)
    lines = ['<record>']
    for k, v in r.items():
        lines.append(f'  <{k}>{v}</{k}>' if not isinstance(v, (list, dict)) else f'  <{k}>{v}</{k}>')
    return '\n'.join(lines + ['</record>'])

def r_table(m):
    return ('| migration_id | backend | status | blocking_test | retry_count |\n'
            '|---|---|---|---|---|\n'
            f"| {m['migration_id']} | {m['backend']} | {m['status']} | {m['blocking_test']} | {m['retry_count']} |")

def r_triples(m):
    s, p, o = m['relation']
    return '\n'.join([f'(migration-01947, has_id, 01947)', f'(migration-01947, has_backend, PostgreSQL)',
                       f'(migration-01947, has_status, blocked)', f'({s}, {p}, {o})',
                       f'(migration-01947, retry_count, 0)', '(migration-01947, rejected, SQLite)'])

def r_compact_csv(m):
    return 'migration_id,backend,status,retry_count,deadline\n01947,PostgreSQL,blocked,,'

RENDERS = {'PROSE': r_prose(MANIFEST), 'KEY_VALUE': r_keyvalue(MANIFEST), 'JSON': r_json(MANIFEST),
           'XML': r_xml(MANIFEST), 'TABLE': r_table(MANIFEST), 'TRIPLES': r_triples(MANIFEST),
           'COMPACT_CSV': r_compact_csv(MANIFEST)}
print(f'{len(RENDERS)} renderings from one manifest.')

## Baseline — fact-survival probes, invalidity marked not hidden

In [ ]:
def probes(text):
    lines = text.splitlines()
    return {
        'ids_distinct': '01947' in text and '01974' in text,
        'zero_kept': any('retry' in ln.lower() and '0' in ln for ln in lines),
        'null_distinct': any(('deadline' in ln and ('null' in ln or 'None' in ln)) or 'no deadline is set' in ln for ln in lines),
        'absent_absent': 'optional_field' not in text,
        'direction': ('depends_on' in text or 'depends on' in text) and text.index('01947') < text.index('suite-A'),
        'dual_source': 'ADR-007' in text and ('staging-log' in text or 'staging log' in text),
        'rationale': 'serialisable transactions' in text,
    }

RESULTS = {name: probes(t) for name, t in RENDERS.items()}
VALID = {n: all(r.values()) for n, r in RESULTS.items()}
# TABLE is a partial view by design: it never claimed the rationale or provenance rows.
VALID['TABLE'] = RESULTS['TABLE']['ids_distinct'] and RESULTS['TABLE']['zero_kept']
print(f"{'probe':14s} {' '.join(f'{n:>11s}' for n in RENDERS)}")
for probe in next(iter(RESULTS.values())):
    print(f"{probe:14s} {' '.join('        [x]' if RESULTS[n][probe] else '        [ ]' for n in RENDERS)}")
print('COMPACT_CSV collapses 0 and null to empty: INVALID for the null-family comparison.')
assert VALID['JSON'] and VALID['PROSE'] and VALID['KEY_VALUE']
assert not RESULTS['COMPACT_CSV']['null_distinct']

## Intervention 1 — same facts, different token structure

Synthetic token estimate (chars // 4): never presented as a provider count.

In [ ]:
print(f"{'form':11s} {'chars':>5s} {'bytes':>5s} {'tok~':>5s} {'rep-names':>9s} {'nest':>4s}")
for name, t in RENDERS.items():
    rep_names = t.count('migration_id') + t.count('backend') + t.count('status')
    nest = max((len(ln) - len(ln.lstrip())) // 1 for ln in t.splitlines()) if '\n' in t else 0
    print(f'{name:11s} {len(t):5d} {len(t.encode()):5d} {len(t) // 4:5d} {rep_names:9d} {nest:4d}')
assert len(RENDERS['JSON']) != len(RENDERS['XML'])
print('JSON vs XML with fields held constant tests syntax; prose vs table tests organisation.')

## Intervention 2 — operation-relative affordances, no winner

In [ ]:
# lookup_steps: deterministic scan cost to the identifier (parse step included for JSON/XML).
lookup = {'PROSE': 4, 'KEY_VALUE': 5, 'JSON': 2, 'XML': 3, 'TABLE': 3, 'TRIPLES': 4}
# explanation: causal/qualifying clauses preserved (full rationale + second reason + qualification).
explain = {'PROSE': 3, 'KEY_VALUE': 1, 'JSON': 1, 'XML': 1, 'TABLE': 0, 'TRIPLES': 0}
# traversal: relation direction explicitly encoded with ordered endpoints.
traverse = {'PROSE': True, 'KEY_VALUE': False, 'JSON': True, 'XML': False, 'TABLE': False, 'TRIPLES': True}
# update locality: lines changed flipping status blocked -> ready.
update_spans = {'PROSE': 2, 'KEY_VALUE': 1, 'JSON': 1, 'XML': 3, 'TABLE': 1, 'TRIPLES': 1}
lookup_w = min(lookup, key=lookup.get)
explain_w = max(explain, key=explain.get)
print(f'lookup winner: {lookup_w}; explanation winner: {explain_w}; traversal explicit: {[k for k, v in traverse.items() if v]}')
assert lookup_w == 'JSON' and explain_w == 'PROSE' and lookup_w != explain_w
assert min(update_spans.values()) < update_spans['PROSE']
print('Divergence preserved: different operations favour different forms. No global ranking computed.')

## Intervention 3 — update locality and round-trip stability

Flip status blocked to ready; count changed lines per serialisation. Then render, parse, update, render for 1, 3, and 5 cumulative edits on JSON, verifying the manifest each time. Machine ledger only.

In [ ]:
def change_status(text, old='blocked', new='ready'):
    a, b = text.splitlines(), text.replace(old, new).splitlines()
    return sum(1 for x, y in zip(a, b) if x != y) + abs(len(a) - len(b))

print('status-flip lines changed:', {k: change_status(v) for k, v in
      [('JSON', RENDERS['JSON']), ('XML', RENDERS['XML']), ('KEY_VALUE', RENDERS['KEY_VALUE'])]})
assert change_status(RENDERS['JSON']) == 1

state = record_dict(MANIFEST)
edits_done = 0
for round_n in (1, 2, 2):
    for _ in range(round_n):
        text = _json.dumps(state, indent=1)
        state = _json.loads(text)
        state['status'] = 'ready' if state['status'] == 'blocked' else 'blocked'
        edits_done += 1
    text = _json.dumps(state, indent=1)
    ok = all(v in text for v in ('01947', 'PostgreSQL', 'serialisable transactions', 'ADR-007'))
    print(f'after {edits_done} edit(s): manifest intact={ok}')
    assert ok
print('Parseability never votes on comprehension: separate ledgers.')

## Observation — hybrid, labelled and unoffered as winner

In [ ]:
hybrid = {'typed_state': ['migration_id=01947', 'status=blocked'],
          'rationale': 'SQLite loses serialisable transactions under batch writes',
          'table': '| migration_id | status |\n| 01947 | blocked |',
          'relations': ['(migration-01947, depends_on, suite-A)'],
          'evidence_ref': 'artifact://compiler-tail@v4'}
print('HYPOTHESISED TASK-SHAPED HYBRID:', list(hybrid))
assert 'rationale' in hybrid and 'evidence_ref' in hybrid
print('Each part names the operation it serves. Not a winner: a hypothesis about matching form to type.')

## Try it

1. Drop `backend_sources` from the JSON renderer and watch the dual-source probe fail while lookup is untouched.
2. Swap the relation endpoints in TRIPLES and confirm the direction probe — not the lookup table — catches it.
3. Render COMPACT_CSV with explicit `null` and confirm it graduates from INVALID to valid-but-sparse.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print(probes(RENDERS['TABLE']))

## What this demonstrates

- Information can remain factually identical while its representation and token-level encoding change.
- Representation and serialisation are different variables: JSON-vs-XML moves syntax; prose-vs-table moves organisation.
- Structural suitability is operation-relative: lookup favours the record, explanation favours prose, and the divergence is the finding.

## What this does not demonstrate

- That JSON beats prose, tables beat records, or XML is better or worse than JSON.
- That structure grants authority, or that parseability implies model comprehension.
- That fewer tokens imply better reasoning, or that hybrid context is universally superior.
- That deterministic rendering predicts real-model preference.

## Connection to the chapter

Representation can preserve conflicting facts perfectly, and still decide nothing between them:

> Representation can preserve conflicting facts perfectly. It still cannot decide which source is allowed to govern behaviour.

That is Chapter 19: Who Gets to Be Right?